# 🛰️ Secure Space Image Transmission System
> **AES-256 Encryption · SHA-256 Integrity · Attack Simulation · Visual Dashboard**

Run each cell **in order**. That's it — no manual config needed.

In [ ]:
# ── Cell 1: Install dependencies ─────────────────────────────────────────────
!pip install pycryptodome pillow matplotlib numpy --quiet

In [ ]:
# ── Cell 2: Write all source files to disk ────────────────────────────────────
import os

# ---------- encryption.py ----------
encryption_src = '''
import os
from Crypto.Cipher import AES
from Crypto.Util.Padding import pad

def generate_key_iv():
    """Generate a random AES-256 key and 128-bit IV."""
    key = os.urandom(32)
    iv  = os.urandom(16)
    return key, iv

def encrypt_image(image_bytes, key, iv):
    """Encrypt image bytes with AES-CBC + PKCS7 padding."""
    cipher    = AES.new(key, AES.MODE_CBC, iv)
    padded    = pad(image_bytes, AES.block_size)
    return cipher.encrypt(padded)

def save_encrypted_file(encrypted_bytes, path):
    """Save cipher-text to a .bin file."""
    with open(path, "wb") as f:
        f.write(encrypted_bytes)
'''
with open('encryption.py', 'w') as f: f.write(encryption_src)

# ---------- decryption.py ----------
decryption_src = '''
from Crypto.Cipher import AES
from Crypto.Util.Padding import unpad

def decrypt_image(encrypted_bytes, key, iv):
    """Decrypt AES-CBC cipher-text and strip PKCS7 padding."""
    try:
        cipher = AES.new(key, AES.MODE_CBC, iv)
        raw    = cipher.decrypt(encrypted_bytes)
        return unpad(raw, AES.block_size)
    except (ValueError, KeyError) as e:
        raise ValueError(f"Decryption failed: {e}") from e

def load_encrypted_file(path):
    """Load encrypted bytes from disk."""
    with open(path, "rb") as f:
        return f.read()
'''
with open('decryption.py', 'w') as f: f.write(decryption_src)

# ---------- integrity.py ----------
integrity_src = '''
import hashlib

def compute_sha256(data):
    """Return SHA-256 hex digest of data."""
    return hashlib.sha256(data).hexdigest()

def verify_integrity(original_hash, received_data):
    """Compare original hash against hash of received data."""
    received_hash = compute_sha256(received_data)
    return (original_hash == received_hash), received_hash
'''
with open('integrity.py', 'w') as f: f.write(integrity_src)

# ---------- transmission.py ----------
transmission_src = '''
import os, random, time

def _tamper_data(data, num_bytes=8):
    payload = bytearray(data)
    for _ in range(num_bytes):
        payload[random.randint(0, len(payload)-1)] = random.randint(0,255)
    return bytes(payload)

def _inject_noise(data, noise_level=0.001):
    payload   = bytearray(data)
    num_noisy = max(1, int(len(payload)*noise_level))
    for idx in random.sample(range(len(payload)), num_noisy):
        payload[idx] ^= random.randint(1,255)
    return bytes(payload)

def transmit(data, attack="none", latency_ms=300):
    time.sleep(latency_ms/1000)
    received, altered = data, 0
    if attack == "tamper":
        num = max(8, len(data)//1000)
        received, altered = _tamper_data(data, num), num
    elif attack == "noise":
        received = _inject_noise(data, 0.002)
        altered  = sum(1 for a,b in zip(data,received) if a!=b)
    return received, {"original_size": len(data), "received_size": len(received),
                      "attack_applied": attack, "latency_ms": latency_ms,
                      "bytes_altered": altered}
'''
with open('transmission.py', 'w') as f: f.write(transmission_src)

# ---------- utils.py ----------
utils_src = '''
import io, os, datetime, sys, time
import numpy as np
from PIL import Image

LOG_FILE = "transmission_log.txt"

def log(message, level="INFO"):
    ts    = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    entry = f"[{ts}] [{level}] {message}"
    print(entry)
    with open(LOG_FILE, "a") as f: f.write(entry+"\\n")

def clear_log():
    with open(LOG_FILE, "w") as f:
        f.write(f"=== Secure Space Log ===\\nStarted: {datetime.datetime.now()}\\n\\n")

def load_image_bytes(path):
    with open(path, "rb") as f: return f.read()

def bytes_to_pil(b):
    return Image.open(io.BytesIO(b))

def save_image_bytes(b, path):
    with open(path, "wb") as f: f.write(b)

def encrypted_preview_array(enc, height=128, width=256):
    needed = height*width
    raw    = np.frombuffer(enc[:needed], dtype=np.uint8)
    if len(raw) < needed: raw = np.pad(raw, (0, needed-len(raw)))
    return raw.reshape((height, width))

def human_size(n):
    if n < 1024: return f"{n} B"
    elif n < 1024**2: return f"{n/1024:.1f} KB"
    return f"{n/1024**2:.2f} MB"

def size_comparison(orig, enc):
    o,e = len(orig), len(enc)
    return {"original": human_size(o), "encrypted": human_size(e),
            "overhead_pct": round((e-o)/o*100,2)}

def progress_bar(label, steps=20, char="█"):
    for i in range(steps+1):
        bar = char*i + "░"*(steps-i)
        sys.stdout.write(f"\\r  {label}: [{bar}] {int(i/steps*100)}%")
        sys.stdout.flush()
        time.sleep(0.02)
    print()
'''
with open('utils.py', 'w') as f: f.write(utils_src)

print("✅ All source files written successfully!")

In [ ]:
# ── Cell 3: Upload your satellite image ───────────────────────────────────────
from google.colab import files
uploaded = files.upload()
IMAGE_PATH = list(uploaded.keys())[0]
print(f"\n✅ Image ready: {IMAGE_PATH}")

In [ ]:
# ── Cell 4: Choose attack mode ────────────────────────────────────────────────
# Options: "none"  |  "tamper"  |  "noise"
ATTACK_MODE = "none"   # ← change me!
print(f"Attack mode set to: [{ATTACK_MODE.upper()}]")

In [ ]:
# ── Cell 5: Run the full pipeline ─────────────────────────────────────────────
import importlib, sys

# Force reimport in case of re-runs
for mod in ['encryption','decryption','integrity','transmission','utils']:
    if mod in sys.modules: del sys.modules[mod]

from encryption   import generate_key_iv, encrypt_image, save_encrypted_file
from decryption   import decrypt_image
from integrity    import compute_sha256, verify_integrity
from transmission import transmit
from utils        import *
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

# ── Constants
ENC_PATH = "encrypted_image.bin"
DEC_PATH = "decrypted_image.png"
BG,PANEL,ACCENT = "#0a0e1a","#111827","#00d4ff"
SAFE_CLR,WARN_CLR,TEXT,MUTED = "#00ff99","#ff4466","#e0e8ff","#556080"

clear_log()
print("╔" + "═"*58 + "╗")
print("║   🛰️  SECURE SPACE IMAGE TRANSMISSION SYSTEM          ║")
print("╚" + "═"*58 + "╝\n")

# ── STAGE 1: Encrypt ──────────────────────────────────────────────────────────
print("\n" + "═"*60)
print("  🛰️  STAGE 1 – ENCRYPTION")
print("═"*60)

progress_bar("Loading image   ", 10)
image_bytes = load_image_bytes(IMAGE_PATH)
log(f"Image loaded: {IMAGE_PATH} ({human_size(len(image_bytes))})")

progress_bar("Generating keys ", 10)
key, iv = generate_key_iv()
log("AES-256 key and IV generated.")

progress_bar("Encrypting      ", 20)
encrypted_bytes = encrypt_image(image_bytes, key, iv)
save_encrypted_file(encrypted_bytes, ENC_PATH)
log(f"Encrypted → {ENC_PATH} ({human_size(len(encrypted_bytes))})", "SUCCESS")

sizes = size_comparison(image_bytes, encrypted_bytes)
print(f"\n  📦 Original : {sizes['original']}")
print(f"  🔒 Encrypted: {sizes['encrypted']}  (+{sizes['overhead_pct']}% overhead)")

original_hash = compute_sha256(encrypted_bytes)
log(f"Pre-TX SHA-256: {original_hash[:16]}…")

# ── STAGE 2: Transmit ─────────────────────────────────────────────────────────
print("\n" + "═"*60)
print("  📡  STAGE 2 – TRANSMISSION")
print("═"*60)
if ATTACK_MODE != "none": print(f"\n  ⚠️  Attack: [{ATTACK_MODE.upper()}]")

progress_bar("Transmitting    ", 25)
received_bytes, meta = transmit(encrypted_bytes, attack=ATTACK_MODE, latency_ms=400)

print(f"\n  Sent    : {human_size(meta['original_size'])}")
print(f"  Recv    : {human_size(meta['received_size'])}")
print(f"  Delay   : {meta['latency_ms']} ms")
if meta['bytes_altered']: print(f"  💥 Altered: {meta['bytes_altered']} bytes")
log(f"Transmission done. Attack={ATTACK_MODE}, Altered={meta['bytes_altered']}")

# ── STAGE 3: Integrity ────────────────────────────────────────────────────────
print("\n" + "═"*60)
print("  🔍  STAGE 3 – INTEGRITY CHECK")
print("═"*60)

progress_bar("Verifying hash  ", 15)
is_intact, received_hash = verify_integrity(original_hash, received_bytes)

print(f"\n  Original : {original_hash[:32]}…")
print(f"  Received : {received_hash[:32]}…")

if is_intact:
    print("\n  ✅  STATUS: DATA INTACT")
    log("Integrity PASSED", "SUCCESS")
else:
    print("\n  ❌  STATUS: DATA COMPROMISED")
    log("Integrity FAILED — data tampered!", "ERROR")

# ── STAGE 4: Decrypt ──────────────────────────────────────────────────────────
print("\n" + "═"*60)
print("  🔓  STAGE 4 – DECRYPTION")
print("═"*60)

decrypted_bytes = None
progress_bar("Decrypting      ", 20)
try:
    decrypted_bytes = decrypt_image(received_bytes, key, iv)
    save_image_bytes(decrypted_bytes, DEC_PATH)
    log(f"Decryption successful → {DEC_PATH}", "SUCCESS")
except ValueError as e:
    print(f"\n  ❌  {e}")
    log(f"Decryption failed: {e}", "ERROR")

# ── STAGE 5: Dashboard ────────────────────────────────────────────────────────
print("\n" + "═"*60)
print("  📊  RENDERING DASHBOARD")
print("═"*60)

def style_ax(ax, title):
    ax.set_facecolor(PANEL)
    ax.set_title(title, color=ACCENT, fontsize=9, fontfamily='monospace', pad=6)
    for s in ax.spines.values(): s.set_edgecolor(MUTED)

fig = plt.figure(figsize=(18, 11), facecolor=BG)
fig.suptitle("🛰️  SECURE SPACE IMAGE TRANSMISSION SYSTEM",
             color=ACCENT, fontsize=17, fontweight='bold',
             fontfamily='monospace', y=0.97)

gs = gridspec.GridSpec(3, 4, figure=fig, hspace=0.55, wspace=0.35,
                       left=0.05, right=0.97, top=0.92, bottom=0.06)

# Original
ax1 = fig.add_subplot(gs[0:2, 0:2])
style_ax(ax1, '[ ORIGINAL SATELLITE IMAGE ]')
orig_img = bytes_to_pil(image_bytes)
ax1.imshow(orig_img); ax1.axis('off')
ax1.text(0.02,0.02,f"Size: {human_size(len(image_bytes))}  |  {orig_img.size[0]}×{orig_img.size[1]} px",
         transform=ax1.transAxes, color=MUTED, fontsize=7, fontfamily='monospace', va='bottom')

# Encrypted preview
ax2 = fig.add_subplot(gs[0, 2])
style_ax(ax2, '[ ENCRYPTED DATA PREVIEW ]')
enc_arr = encrypted_preview_array(encrypted_bytes, 128, 192)
ax2.imshow(enc_arr, cmap='plasma', aspect='auto', interpolation='nearest'); ax2.axis('off')
ax2.text(0.02,0.02,f"AES-256-CBC  |  {human_size(len(encrypted_bytes))}",
         transform=ax2.transAxes, color=MUTED, fontsize=7, fontfamily='monospace', va='bottom')

# Decrypted
ax3 = fig.add_subplot(gs[1, 2])
style_ax(ax3, '[ DECRYPTED IMAGE ]')
if decrypted_bytes:
    try:
        ax3.imshow(bytes_to_pil(decrypted_bytes)); ax3.axis('off')
    except:
        ax3.text(0.5,0.5,'⚠️\nCORRUPTED',ha='center',va='center',
                 color=WARN_CLR,fontsize=12,fontfamily='monospace',transform=ax3.transAxes)
        ax3.axis('off')
else:
    ax3.text(0.5,0.5,'❌\nDECRYPTION\nFAILED',ha='center',va='center',
             color=WARN_CLR,fontsize=11,fontfamily='monospace',transform=ax3.transAxes)
    ax3.axis('off')

# Integrity status
ax4 = fig.add_subplot(gs[0, 3])
ax4.set_facecolor(PANEL); ax4.axis('off')
sc = SAFE_CLR if is_intact else WARN_CLR
ax4.text(0.5,0.62,'✅' if is_intact else '❌',ha='center',va='center',
         fontsize=32,transform=ax4.transAxes)
ax4.text(0.5,0.30,'DATA\nINTACT' if is_intact else 'DATA\nCOMPROMISED',
         ha='center',va='center',fontsize=14,fontweight='bold',
         color=sc,fontfamily='monospace',transform=ax4.transAxes)
ax4.set_title('[ INTEGRITY STATUS ]',color=ACCENT,fontsize=9,fontfamily='monospace',pad=6)
for s in ax4.spines.values(): s.set_edgecolor(MUTED)

# Transmission meta
ax5 = fig.add_subplot(gs[1, 3])
ax5.set_facecolor(PANEL); ax5.axis('off')
ax5.set_title('[ TRANSMISSION META ]',color=ACCENT,fontsize=9,fontfamily='monospace',pad=6)
info = [f"  Attack  : {meta['attack_applied'].upper()}",
        f"  Latency : {meta['latency_ms']} ms",
        f"  Sent    : {human_size(meta['original_size'])}",
        f"  Recv    : {human_size(meta['received_size'])}",
        f"  Altered : {meta['bytes_altered']} bytes"]
for i,line in enumerate(info):
    col = WARN_CLR if meta['bytes_altered']>0 and 'Altered' in line else TEXT
    ax5.text(0.05,0.82-i*0.16,line,transform=ax5.transAxes,
             color=col,fontsize=8.5,fontfamily='monospace')
for s in ax5.spines.values(): s.set_edgecolor(MUTED)

# Size comparison
ax6 = fig.add_subplot(gs[2, :2])
style_ax(ax6, '[ SIZE COMPARISON ]')
vals = [len(image_bytes)/1024, len(encrypted_bytes)/1024]
bars = ax6.bar(['Original','Encrypted'], vals, color=[ACCENT,'#ff9f43'], width=0.4, edgecolor=MUTED, linewidth=0.8)
ax6.set_ylabel('Size (KB)', color=TEXT, fontsize=8, fontfamily='monospace')
ax6.tick_params(colors=TEXT, labelsize=8)
[ax6.text(b.get_x()+b.get_width()/2, b.get_height()+max(vals)*0.02,
           f"{v:.1f} KB", ha='center', color=TEXT, fontsize=8, fontfamily='monospace')
 for b,v in zip(bars,vals)]
ax6.set_facecolor(PANEL); ax6.tick_params(axis='x',colors=TEXT)
for s in ax6.spines.values(): s.set_edgecolor(MUTED)
ax6.set_ylim(0, max(vals)*1.18)

# Byte histogram
ax7 = fig.add_subplot(gs[2, 2:])
style_ax(ax7, '[ ENCRYPTED BYTE DISTRIBUTION ]')
bv = np.frombuffer(encrypted_bytes[:8192], dtype=np.uint8)
ax7.hist(bv, bins=64, color=ACCENT, alpha=0.85, edgecolor='none')
ax7.set_xlabel('Byte value (0-255)', color=TEXT, fontsize=8, fontfamily='monospace')
ax7.set_ylabel('Frequency', color=TEXT, fontsize=8, fontfamily='monospace')
ax7.tick_params(colors=TEXT, labelsize=7)
for s in ax7.spines.values(): s.set_edgecolor(MUTED)
ax7.set_facecolor(PANEL)

plt.savefig('dashboard.png', dpi=140, bbox_inches='tight', facecolor=BG)
plt.show()
log('Dashboard rendered → dashboard.png', 'SUCCESS')

print("\n╔" + "═"*58 + "╗")
print("║   ✅  PIPELINE COMPLETE                                  ║")
print("╚" + "═"*58 + "╝")

In [ ]:
# ── Cell 6 (optional): Download outputs ──────────────────────────────────────
from google.colab import files
files.download('dashboard.png')
files.download('decrypted_image.png')
files.download('encrypted_image.bin')
files.download('transmission_log.txt')